# 03 — Región de prueba de Lima (marzo de 2017)

El examen pide construir **nuestra propia región de prueba en Perú** con **dos fuentes independientes**:

| Fuente | Qué es | De dónde | Estado |
|---|---|---|---|
| 1. Imagen de radar | Sentinel-1 GRD (VV/VH) sobre Lima, 17-mar-2017 | Google Earth Engine (`COPERNICUS/S1_GRD`) | **Parte A (este notebook)** |
| 2. Etiquetas | Polígonos de inundación de **EMSN038** (Copernicus EMS) | Geodatabase descargada desde el navegador | Parte B (después) |

Después se cortan ambas en **chips de 512×512** (igual que Sen1Floods11) y se evalúa el modelo final **sin reentrenar**.

### ¿Por qué Earth Engine y no descargar la imagen cruda?

La imagen cruda de Copernicus/ASF (≈ 1 GB) viene **sin procesar**. Habría que calibrarla, quitarle el ruido térmico, corregir el relieve y pasarla a dB con el programa SNAP. Earth Engine ya entrega la colección `COPERNICUS/S1_GRD` **con exactamente ese procesamiento**, y **Sen1Floods11 se construyó con esa misma colección**. Así nuestra imagen de Lima queda en el mismo formato que los chips de entrenamiento: VV/VH en dB, 10 m, EPSG:4326. Es la misma escena GRD que está en Copernicus y en ASF; solo cambia quién la procesa.

> Este notebook **no necesita GPU**. Se puede correr en otra pestaña de Colab mientras `02_modelado` entrena.

## A0. Antes de empezar (una sola vez)

1. Registrarse en Earth Engine con la cuenta de Google: https://code.earthengine.google.com/register
   - Elegir **uso no comercial** (académico) y **crear un proyecto de Google Cloud** (es gratis).
2. ✅ Hecho: proyecto `ee-tarea2-floods`, registrado para uso no comercial (nivel Comunidad).

In [ ]:
import ee

PROYECTO = 'ee-tarea2-floods'      # ID del proyecto de Earth Engine (registrado, nivel Comunidad)
ee.Authenticate()             # abre un enlace para autorizar con tu cuenta de Google
ee.Initialize(project=PROYECTO)
print('Earth Engine listo')

## A1. Zona de estudio

Un rectángulo sobre Lima que cubre los ríos **Rímac** y **Chillón** y los distritos de EMSN038: Carapongo, Huachipa, Cajamarquilla, San Martín de Porres, Comas, Puente Piedra y Carabayllo.

Mide ≈ 44 × 39 km. A 10 m por píxel son ≈ 4,400 × 3,900 píxeles, que luego se cortan en chips de 512.

In [ ]:
AOI = ee.Geometry.Rectangle([-77.20, -12.10, -76.80, -11.75])     # [oeste, sur, este, norte] en grados

## A2. ¿Qué imágenes Sentinel-1 hay sobre Lima en esas fechas?

El huaico fue entre el 16 y el 19 de marzo de 2017, y las imágenes ópticas de EMSN038 son del 18 y 21 de marzo. Listamos todas las pasadas de Sentinel-1 entre el 10 y el 25 de marzo para confirmar la elección.

Lo que ya se sabe (de la investigación previa):
- **17-mar-2017, Sentinel-1B, órbita descendente, track 171, VV+VH** → la elegida.
- Las del 18 y 24 de marzo son **solo VV**: no sirven, porque la red necesita las dos bandas.

In [ ]:
coleccion = (ee.ImageCollection('COPERNICUS/S1_GRD')
             .filterBounds(AOI)
             .filterDate('2017-03-10', '2017-03-26')
             .filter(ee.Filter.eq('instrumentMode', 'IW')))

info = coleccion.getInfo()['features']
import pandas as pd
escenas = pd.DataFrame([{
    'id': f['id'].split('/')[-1],
    'fecha': f['id'].split('/')[-1].split('_')[4][:8],
    'satelite': f['properties']['platform_number'],
    'orbita': f['properties']['orbitProperties_pass'],
    'track': f['properties']['relativeOrbitNumber_start'],
    'bandas': ','.join(b for b in f['properties']['transmitterReceiverPolarisation']),
} for f in info]).sort_values('fecha')
escenas

## A3. Elegir la escena del 17 de marzo (VV + VH)

Si la escena principal cubre la zona en dos "pedazos" (dos imágenes seguidas de la misma pasada), se unen con `mosaic()`.

In [ ]:
s1 = (coleccion
      .filterDate('2017-03-17', '2017-03-18')
      .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VH'))
      .filter(ee.Filter.eq('relativeOrbitNumber_start', 171)))
print('Imágenes del 17-mar con VV+VH, track 171:', s1.size().getInfo())
print([i['id'] for i in s1.getInfo()['features']])

lima = s1.select(['VV', 'VH']).mosaic().clip(AOI).toFloat()
print('Cobertura de la zona (%):', round(100 * lima.select('VV').mask().reduceRegion(
    ee.Reducer.mean(), AOI, scale=100).get('VV').getInfo(), 1))

## A4. Exportar a Drive

- **EPSG:4326 y 10 m**, igual que Sen1Floods11.
- Los píxeles sin dato se exportan como **−9999** (GeoTIFF no guarda NaN de forma fiable desde Earth Engine). En la Parte C se convierten a NaN.
- Se guarda en `MyDrive/tarea2_floods_lima/`. **Tarda unos 5–15 minutos**. Se puede seguir el avance en https://code.earthengine.google.com/tasks o con la celda siguiente.

In [ ]:
tarea = ee.batch.Export.image.toDrive(
    image=lima.unmask(-9999),
    description='S1_Lima_20170317',
    folder='tarea2_floods_lima',
    fileNamePrefix='S1_Lima_20170317_VV_VH_dB',
    region=AOI,
    crs='EPSG:4326',
    scale=10,
    maxPixels=1e9,
)
tarea.start()
print('Exportación iniciada')

In [ ]:
tarea.status()['state']      # READY → RUNNING → COMPLETED (volver a ejecutar para actualizar)

## A5. Revisar la imagen exportada

> **Desde aquí empieza cada sesión nueva.** Las celdas A0–A4 (Earth Engine) se corren **una sola vez**: la imagen ya queda guardada en Drive.

Cuando el estado sea `COMPLETED`, montamos Drive y miramos la imagen. Si el archivo pesa mucho, Earth Engine puede partirlo en varios `.tif`; la celda los lista.

**Qué esperar:** el mar (oeste) y los ríos se ven **oscuros** (agua lisa); la ciudad se ve **muy brillante** (doble rebote en los edificios); los cerros tienen zonas brillantes y sombras oscuras del lado opuesto al satélite.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
!pip install -q rasterio pyogrio
import rasterio

LIMA = '/content/drive/MyDrive/tarea2_floods_lima'
archivos = sorted(glob.glob(f'{LIMA}/S1_Lima_20170317*.tif'))
for a in archivos:
    print(os.path.basename(a), f'{os.path.getsize(a) / 1e6:.0f} MB')

In [ ]:
with rasterio.open(archivos[0]) as src:
    print('Tamaño:', src.width, '×', src.height, '| bandas:', src.count, '| CRS:', src.crs)
    print('Tamaño de píxel (grados):', src.res)
    img = src.read().astype('float32')
    TRANSFORM, CRS, H, W = src.transform, src.crs, src.height, src.width    # la grilla de Lima: las máscaras se dibujan sobre ella
img[img == -9999] = np.nan
sin_imagen = np.isnan(img).any(axis=0)
print('% sin dato:', round(100 * sin_imagen.mean(), 1))

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
axes[0].imshow(img[0], cmap='gray', vmin=-25, vmax=0);  axes[0].set_title('VV (dB) · 17-mar-2017')
axes[1].imshow(img[1], cmap='gray', vmin=-30, vmax=-5); axes[1].set_title('VH (dB)')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

# Parte B. Etiquetas EMSN038

El geodatabase de Copernicus (`EMSN038_UTM18S_v0.gdb`) trae ≈ 40 capas. Ya revisamos su catálogo; estas son las que nos sirven:

| Capa | Grupo | Qué es | Uso aquí |
|---|---|---|---|
| `Potential_Maximum_flood` | RRM (riesgo) | **Huella máxima** de la inundación: zonas con lodo, humedad y restos del huaico, vistas en imágenes ópticas del 18 y 21 de marzo | Referencia 1 (la "oficial") |
| `Riverbank` | RRM (riesgo) | Orillas del río afectadas | Referencia 2 (vemos qué geometría tiene) |
| `WatrcrsA` | Referencia | **Cauce** de los ríos (polígonos del mapa base, antes del evento) | Referencia 3: dónde **debería** haber agua del río |
| `EMSN038_AOI` | Referencia | Zona que Copernicus analizó | Fuera de ella la máscara es **−1** (no se evalúa) |

Ojo con los nombres: `Potential_Maximum_flood` lleva **M mayúscula** y `Riverbank` va en **singular**.

**Dato para el informe:** el geodatabase también trae rásters de **cambio en Sentinel-1** entre el 5 y el 17 de marzo (`T05032017_to_17032017_Change_VV/VH`). Copernicus intentó ver el agua con radar y **no encontró agua estancada**: por eso su capa de inundación sale de imágenes ópticas.

### B0. Subir el geodatabase (una sola vez)

El zip original pesa 2.3 GB porque trae esos rásters. Armamos uno liviano **solo con las capas vectoriales** (98 MB): `tarea2/ems038/EMSN038_vectores.zip`.

➡️ Súbelo a `MyDrive/tarea2_floods_lima/` (arrastrándolo a Drive en el navegador).

In [ ]:
import shutil, pyogrio
import geopandas as gpd

ZIP = f'{LIMA}/EMSN038_vectores.zip'           # si falla, sube el zip completo y cambia el nombre aquí
shutil.copy(ZIP, '/content/emsn038.zip')        # leer desde el disco local es más rápido que desde Drive
GDB = '/vsizip//content/emsn038.zip/EMSN038_UTM18S_v0.gdb'

capas = pd.DataFrame(pyogrio.list_layers(GDB), columns=['capa', 'geometria'])
capas

### B1. Leer las 4 capas

Vienen en **EPSG:32618** (UTM zona 18 Sur, en metros). Así podemos medir áreas en km² antes de reproyectar.

In [ ]:
def leer(capa):
    g = gpd.read_file(GDB, layer=capa)
    print(f'{capa:25s} {len(g):5d} elementos · {g.geom_type.value_counts().to_dict()} · {g.crs.to_string()} · '
          f'{g.area.sum() / 1e6:.2f} km²')
    return g

pmf   = leer('Potential_Maximum_flood')
rb    = leer('Riverbank')
cauce = leer('WatrcrsA')
aoi   = leer('EMSN038_AOI')

¿Qué columnas traen? Sirve para saber si hay tipos distintos (por ejemplo, "inundado" frente a "lodo") o fechas.

In [ ]:
pmf.drop(columns='geometry').head()

In [ ]:
rb.drop(columns='geometry').head()

In [ ]:
for nombre, g in [('pmf', pmf), ('rb', rb), ('cauce', cauce), ('aoi', aoi)]:
    print(f'\n=== {nombre} ===')
    for col in g.columns.drop('geometry'):
        if g[col].dtype == object and g[col].nunique() <= 15:
            print(col, '→', g[col].value_counts(dropna=False).to_dict())

**Resultado (16-oct):** en la huella, `source` = `'river'` (2 polígonos: el cauce del Rímac y del Chillón) y `'maximun height'` (11 polígonos: hasta dónde llegó el huaico). `Riverbank` son esos mismos 11 polígonos. Por eso en B3 la huella se separa en **río** y **huaico**: son dos cosas distintas para el radar (agua corriendo frente a lodo).

### B2. ¿Nuestra imagen cubre todo EMSN038?

El rectángulo que exportamos de Earth Engine es `[-77.20, -12.10, -76.80, -11.75]`. Si una parte de la huella queda fuera, hay que agrandarlo y volver a exportar (A1–A4).

In [ ]:
from shapely.geometry import box
rect = gpd.GeoSeries([box(-77.20, -12.10, -76.80, -11.75)], crs='EPSG:4326').to_crs(pmf.crs).iloc[0]

print('Límites de EMSN038_AOI (lon/lat):', aoi.to_crs('EPSG:4326').total_bounds.round(3))
for nombre, g in [('Huella máxima', pmf), ('Riverbank', rb), ('Cauce', cauce), ('AOI de EMSN038', aoi)]:
    dentro = g.intersection(rect).area.sum() / max(g.area.sum(), 1)
    print(f'{nombre:15s}: {100 * dentro:5.1f} % de su área cae dentro de nuestra imagen')

### B3. Rasterizar: de polígonos a máscara de píxeles

La red da una respuesta **por píxel**, así que los polígonos se "pintan" sobre **la misma grilla** de la imagen (mismo tamaño y misma posición de cada píxel):

1. Si una capa es de **líneas** (por ejemplo, una orilla dibujada como línea), se le da un ancho de 10 m (1 píxel) para que tenga área.
2. Se reproyecta de EPSG:32618 a EPSG:4326 (el sistema de la imagen).
3. `rasterize` pone **1** en los píxeles cuyo centro cae dentro de un polígono y **0** en el resto.
4. Se pone **−1** fuera del AOI de EMSN038 (Copernicus no miró ahí) y donde la imagen no tiene dato. Igual que en Sen1Floods11.

In [ ]:
from rasterio import features

def rasterizar(g):
    g = g[g.geometry.notna() & ~g.geometry.is_empty]
    lineas = g.geom_type.str.contains('Line')
    g = g.set_geometry(g.geometry.where(~lineas, g.geometry.buffer(5)))      # línea → franja de 10 m
    g = g.to_crs(CRS)
    return features.rasterize(((geom, 1) for geom in g.geometry), out_shape=(H, W),
                              transform=TRANSFORM, fill=0, dtype='uint8')

dentro_aoi = rasterizar(aoi).astype(bool)

def a_mascara(capa):
    lab = rasterizar(capa).astype('int8')
    lab[~dentro_aoi | sin_imagen] = -1            # fuera del AOI o sin radar → no se evalúa
    return lab

# La huella tiene 2 tipos de polígono (columna `source`): 'river' (el río) y 'maximun height' (la extensión del huaico).
# Riverbank son esos mismos 11 polígonos 'maximun height', así que no se repite.
MASCARAS = {
    'Huella máxima': a_mascara(pmf),
    'Río (EMSN038)': a_mascara(pmf[pmf['source'] == 'river']),
    'Huaico (EMSN038)': a_mascara(pmf[pmf['source'] == 'maximun height']),
    'Cauce': a_mascara(cauce),
}
pd.DataFrame({k: {'px_validos': int((m != -1).sum()), 'px_agua': int((m == 1).sum()),
                  'pct_agua': 100 * (m == 1).sum() / max((m != -1).sum(), 1)} for k, m in MASCARAS.items()}).T.round(3)

### B4. Guardar las máscaras en Drive

Así la Parte C no necesita volver a leer el geodatabase. Un GeoTIFF de 4 bandas (`int8`, −1/0/1), con la misma grilla que la imagen.

In [ ]:
perfil = {'driver': 'GTiff', 'height': H, 'width': W, 'count': len(MASCARAS), 'dtype': 'int8',
          'crs': CRS, 'transform': TRANSFORM, 'compress': 'deflate'}
with rasterio.open(f'{LIMA}/Lima_mascaras_EMSN038.tif', 'w', **perfil) as dst:
    for i, (nombre, m) in enumerate(MASCARAS.items(), start=1):
        dst.write(m, i)
        dst.set_band_description(i, nombre)
print('Guardado:', f'{LIMA}/Lima_mascaras_EMSN038.tif')

### B5. Mirar las máscaras sobre el radar

Dos acercamientos:
- **Rímac** (Huachipa, Carapongo, Cajamarquilla): ≈ 90 % de la huella según el reporte de EMSN038.
- **Chillón** (Comas, Carabayllo, Puente Piedra).

Las coordenadas de las ventanas son aproximadas: si la huella queda cortada, se ajustan aquí.

In [ ]:
from rasterio.transform import rowcol

ZONAS = {                                   # [oeste, sur, este, norte]
    'Rímac (Huachipa–Carapongo)': [-76.97, -12.04, -76.82, -11.93],
    'Chillón (Comas–Carabayllo)': [-77.12, -11.97, -76.98, -11.80],
}

def ventana(oeste, sur, este, norte):
    """Coordenadas (lon/lat) → cortes de filas y columnas en la imagen."""
    f0, c0 = rowcol(TRANSFORM, oeste, norte)
    f1, c1 = rowcol(TRANSFORM, este, sur)
    return slice(max(f0, 0), min(f1, H)), slice(max(c0, 0), min(c1, W))

def show_mask(ax, lab, title):
    rgb = np.zeros((*lab.shape, 3))
    rgb[lab == 0] = [0.89, 0.84, 0.72]      # no agua (beige)
    rgb[lab == 1] = [0.12, 0.44, 0.70]      # agua / huella (azul)
    rgb[lab == -1] = [0.60, 0.60, 0.60]     # sin dato / fuera del AOI (gris)
    ax.imshow(rgb); ax.set_title(title); ax.axis('off')

for zona, coords in ZONAS.items():
    f, c = ventana(*coords)
    fig, axes = plt.subplots(1, 1 + len(MASCARAS), figsize=(5 * (1 + len(MASCARAS)), 5))
    axes[0].imshow(img[0][f, c], cmap='gray', vmin=-25, vmax=0); axes[0].set_title(f'{zona} · VV'); axes[0].axis('off')
    for ax, (nombre, m) in zip(axes[1:], MASCARAS.items()):
        show_mask(ax, m[f, c], nombre)
    plt.tight_layout(); plt.show()

**Qué mirar:** ¿la huella (azul) coincide con zonas **oscuras** del VV? Si la huella cae sobre zonas grises o brillantes, el radar del 17 de marzo **no ve agua ahí** (lodo seco, suelo removido o el agua ya se fue). Eso es justo lo que dice el reporte de Copernicus, y anticipa un IoU bajo contra la huella.

# Parte C. Chips y evaluación del modelo (sin reentrenar)

- La imagen de Lima se corta en **chips de 512×512**, igual que Sen1Floods11. Los bordes que sobran se rellenan con "sin dato".
- Cada chip se normaliza con **`norm_stats.json` de train** (lo que hizo `02_modelado`) y pasa por la red.
- Se evalúan **6 modelos ya entrenados**: las 2 recetas × 3 semillas (decidido antes de mirar Lima, sección 20 de `02_modelado`):
  - **Solo train** = `E3_bce_dice` (+ `_s1`, `_s2`).
  - **Train + valid** = `FINAL_E3_bce_dice_train_valid` (+ `_s1`, `_s2`) → el modelo final.
- Línea base: **Otsu** sobre la banda VH de la escena.
- **No se reentrena ni se ajusta nada con Lima.** Es una prueba de generalización.

> GPU opcional: con CPU tarda unos minutos más.

### C0. Las piezas de `02_modelado` que se necesitan

Copiadas **sin cambios**: la U-Net, la normalización y las métricas. (Para el repo final conviene moverlas a un archivo `.py` común.)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

OUT = '/content/drive/MyDrive/tarea2_floods/outputs'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
STATS = json.load(open(f'{OUT}/norm_stats.json'))          # estadísticas de train (no de Lima)
print('Device:', DEVICE, '| normalización de train:', {k: np.round(v, 2).tolist() for k, v in STATS.items()})


def normalize(img):
    """dB crudo → recorta extremos, estandariza por banda y pone 0 donde no hay dato."""
    lo, hi = np.array(STATS['lo'])[:, None, None], np.array(STATS['hi'])[:, None, None]
    mean, std = np.array(STATS['mean'])[:, None, None], np.array(STATS['std'])[:, None, None]
    x = (np.clip(img, lo, hi) - mean) / std
    return np.nan_to_num(x, nan=0.0).astype('float32')


def double_conv(in_ch, out_ch):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False), nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
    )


class AttentionGate(nn.Module):
    """Pesa cada píxel del skip (x) con un valor 0–1 calculado a partir del decoder (g)."""
    def __init__(self, ch, inter):
        super().__init__()
        self.w_g = nn.Sequential(nn.Conv2d(ch, inter, 1, bias=False), nn.BatchNorm2d(inter))
        self.w_x = nn.Sequential(nn.Conv2d(ch, inter, 1, bias=False), nn.BatchNorm2d(inter))
        self.psi = nn.Sequential(nn.Conv2d(inter, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid())

    def forward(self, x, g):
        alpha = self.psi(F.relu(self.w_g(g) + self.w_x(x)))
        return x * alpha


class UNet(nn.Module):
    def __init__(self, in_channels=2, out_channels=1, features=(32, 64, 128, 256), attention=False):
        super().__init__()
        self.down = nn.ModuleList()
        ch = in_channels
        for f in features:
            self.down.append(double_conv(ch, f))
            ch = f
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = double_conv(ch, ch * 2)
        self.up = nn.ModuleList()
        self.dec = nn.ModuleList()
        for f in reversed(features):
            self.up.append(nn.ConvTranspose2d(f * 2, f, kernel_size=2, stride=2))
            self.dec.append(double_conv(f * 2, f))
        self.gates = nn.ModuleList([AttentionGate(f, f // 2) for f in reversed(features)]) if attention else None
        self.head = nn.Conv2d(features[0], out_channels, kernel_size=1)

    def forward(self, x):
        skips = []
        for block in self.down:
            x = block(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        for i, (up, dec, skip) in enumerate(zip(self.up, self.dec, reversed(skips))):
            x = up(x)
            if self.gates is not None:
                skip = self.gates[i](skip, x)
            x = torch.cat([skip, x], dim=1)
            x = dec(x)
        return self.head(x)


class WaterMetrics:
    """Matriz de confusión de la clase agua sobre los píxeles válidos (target != -1)."""
    def __init__(self):
        self.tp = self.fp = self.fn = self.tn = 0

    def update(self, pred, target):
        valid, agua = target != -1, target == 1
        self.tp += int(( pred &  agua & valid).sum())
        self.fp += int(( pred & ~agua & valid).sum())
        self.fn += int((~pred &  agua & valid).sum())
        self.tn += int((~pred & ~agua & valid).sum())

    def compute(self):
        tp, fp, fn, tn = self.tp, self.fp, self.fn, self.tn
        precision, recall = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
        return {
            'iou': tp / max(tp + fp + fn, 1),
            'f1': 2 * precision * recall / max(precision + recall, 1e-9),
            'precision': precision,
            'recall': recall,
            'omision': fn / max(tp + fn, 1),
            'comision': fp / max(tp + fp, 1),
            'accuracy': (tp + tn) / max(tp + fp + fn + tn, 1),
        }


def load_model(name):
    ckpt = torch.load(f'{OUT}/{name}/best.pt', map_location=DEVICE)
    model = UNet(attention=ckpt['config'].get('attention', False)).to(DEVICE)
    model.load_state_dict(ckpt['model'])
    print(f"{name}: época {ckpt['epoch']} · {ckpt['config']}")
    return model.eval()

### C1. Cargar las máscaras y cortar en chips de 512

Si es una sesión nueva, basta con correr A5 (imagen) y esta celda: las máscaras se leen del GeoTIFF guardado en B4.

In [ ]:
with rasterio.open(f'{LIMA}/Lima_mascaras_EMSN038.tif') as src:
    MASCARAS = {src.descriptions[i]: src.read(i + 1) for i in range(src.count)}
print('Máscaras:', list(MASCARAS))

CHIP = 512
NF, NC = -(-H // CHIP), -(-W // CHIP)              # nº de filas y columnas de chips (redondeo hacia arriba)
img_pad = np.full((2, NF * CHIP, NC * CHIP), np.nan, dtype='float32')
img_pad[:, :H, :W] = img                             # los bordes que sobran quedan como "sin dato"

inventario = []
for i in range(NF):
    for j in range(NC):
        f, c = slice(i * CHIP, (i + 1) * CHIP), slice(j * CHIP, (j + 1) * CHIP)
        lab = MASCARAS['Huella máxima'][f, c]
        n_eval = max((lab != -1).sum(), 1)
        inventario.append({'chip': f'Lima_{i}_{j}', 'pct_evaluable': 100 * (lab != -1).mean(),
                           'pct_huella': 100 * (lab == 1).sum() / n_eval,
                           'pct_cauce': 100 * (MASCARAS['Cauce'][f, c] == 1).sum() / n_eval})
inventario = pd.DataFrame(inventario)
print(f'{NF} × {NC} = {len(inventario)} chips · con algo evaluable: {(inventario.pct_evaluable > 0).sum()} · '
      f'con huella: {(inventario.pct_huella > 0).sum()}')
inventario.sort_values('pct_huella', ascending=False).head(10).round(2)

### C2. Predecir toda la escena, chip por chip

Cada chip de 512 pasa por la red (igual que un chip de test de Sen1Floods11) y la respuesta se pega en su lugar. Los chips sin ningún dato de radar (el borde negro de la pasada) se saltan.

In [ ]:
@torch.no_grad()
def predecir_lima(model):
    pred = np.zeros(img_pad.shape[1:], dtype=bool)
    for i in range(NF):
        for j in range(NC):
            x = img_pad[:, i * CHIP:(i + 1) * CHIP, j * CHIP:(j + 1) * CHIP]
            if np.isnan(x).all():
                continue
            t = torch.from_numpy(normalize(x))[None].to(DEVICE)
            pred[i * CHIP:(i + 1) * CHIP, j * CHIP:(j + 1) * CHIP] = (model(t)[0, 0] > 0).cpu().numpy()
    return pred[:H, :W] & ~sin_imagen


SEMILLAS = [42, 1, 2]
RECETAS = {'Solo train': 'E3_bce_dice', 'Train + valid': 'FINAL_E3_bce_dice_train_valid'}     # ELEGIDO y FINAL de 02_modelado

def con_semilla(nombre, s):
    return nombre if s == 42 else f'{nombre}_s{s}'

PRED = {}
for receta, nombre in RECETAS.items():
    for s in SEMILLAS:
        PRED[(receta, s)] = predecir_lima(load_model(con_semilla(nombre, s)))
print('Predicciones listas:', len(PRED))

### C3. Línea base: Otsu sobre VH

Otsu busca **el umbral que mejor separa** el histograma de VH en dos grupos (oscuro = agua, claro = resto). En Sen1Floods11 los umbrales por evento iban de −19.6 a −25.1 dB. Aquí se calcula sobre toda la escena de Lima (incluye el mar, que ayuda a que haya un grupo "agua" claro).

In [ ]:
from skimage.filters import threshold_otsu

vh_valido = img[1][~sin_imagen]
UMBRAL_VH = threshold_otsu(vh_valido)
otsu_lima = (img[1] < UMBRAL_VH) & ~sin_imagen
print(f'Umbral de Otsu en VH: {UMBRAL_VH:.2f} dB · {100 * otsu_lima.mean():.1f} % de la escena marcada como agua')

plt.figure(figsize=(8, 3))
plt.hist(vh_valido[::50], bins=120, color='gray')
plt.axvline(UMBRAL_VH, color='red', label=f'Otsu = {UMBRAL_VH:.1f} dB')
plt.xlabel('VH (dB)'); plt.legend(); plt.title('Histograma de VH en Lima'); plt.show()

### C4. Métricas contra cada referencia

Para cada referencia (huella completa, río y huaico por separado, cauce) y cada modelo, se calcula IoU, precisión, recall, etc., **solo dentro del AOI de EMSN038**. Luego se resume como **media ± desviación** de las 3 semillas.

In [ ]:
filas = []
for nombre_ref, lab in MASCARAS.items():
    evaluable = lab != -1
    m = WaterMetrics(); m.update(otsu_lima, lab)
    filas.append({'referencia': nombre_ref, 'modelo': 'Otsu', 'semilla': None, **m.compute(),
                  'pct_pred_agua': 100 * otsu_lima[evaluable].mean()})
    for (receta, s), pred in PRED.items():
        m = WaterMetrics(); m.update(pred, lab)
        filas.append({'referencia': nombre_ref, 'modelo': receta, 'semilla': s, **m.compute(),
                      'pct_pred_agua': 100 * pred[evaluable].mean()})

lima = pd.DataFrame(filas)
lima.to_csv(f'{OUT}/resultados_lima.csv', index=False)

resumen_lima = (lima.groupby(['referencia', 'modelo'], sort=False)[['iou', 'precision', 'recall', 'comision', 'pct_pred_agua']]
                .agg(['mean', 'std']))
resumen_lima.round(3)

**IoU promedio por chip** (la métrica de los autores de Sen1Floods11, para comparar con test 0.400 y Bolivia 0.366): solo los chips donde hay huella o predicción.

In [ ]:
def iou_por_chip(pred, lab):
    ious = []
    for i in range(NF):
        for j in range(NC):
            f, c = slice(i * CHIP, (i + 1) * CHIP), slice(j * CHIP, (j + 1) * CHIP)
            m = WaterMetrics(); m.update(pred[f, c], lab[f, c])
            if m.tp + m.fp + m.fn > 0:
                ious.append(m.compute()['iou'])
    return np.mean(ious)

for nombre_ref in MASCARAS:
    lab = MASCARAS[nombre_ref]
    for receta in RECETAS:
        vals = [iou_por_chip(PRED[(receta, s)], lab) for s in SEMILLAS]
        print(f'{nombre_ref:14s} · {receta:13s}: IoU por chip {np.mean(vals):.3f} ± {np.std(vals, ddof=1):.3f}')

### C5. Mapa de errores en las dos zonas

Modelo final (`Train + valid`, semilla 42) frente a la huella y frente al cauce, y Otsu frente a la huella. Mismos colores que en `02_modelado`:
- **Azul:** acierto de agua · **Rojo:** agua inventada (comisión) · **Naranja:** agua no vista (omisión) · **Beige:** no agua bien · **Gris:** fuera del AOI o sin dato.

In [ ]:
def show_errores(ax, pred, lab, title):
    rgb = np.zeros((*lab.shape, 3))
    rgb[:] = [0.89, 0.84, 0.72]
    rgb[pred & (lab == 1)] = [0.12, 0.44, 0.70]
    rgb[pred & (lab == 0)] = [0.84, 0.15, 0.16]
    rgb[~pred & (lab == 1)] = [1.00, 0.55, 0.00]
    rgb[lab == -1] = [0.60, 0.60, 0.60]
    ax.imshow(rgb); ax.set_title(title); ax.axis('off')

pred_final = PRED[('Train + valid', 42)]
for zona, coords in ZONAS.items():
    f, c = ventana(*coords)
    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    axes[0].imshow(img[0][f, c], cmap='gray', vmin=-25, vmax=0); axes[0].set_title(f'{zona} · VV'); axes[0].axis('off')
    show_errores(axes[1], pred_final[f, c], MASCARAS['Huella máxima'][f, c], 'Red vs huella máxima')
    show_errores(axes[2], pred_final[f, c], MASCARAS['Cauce'][f, c], 'Red vs cauce')
    show_errores(axes[3], otsu_lima[f, c], MASCARAS['Huella máxima'][f, c], 'Otsu vs huella máxima')
    plt.tight_layout(); plt.show()

### C6. Toda la escena: ¿dónde "ve" agua la red?

Fuera del AOI de EMSN038 no hay etiqueta, pero sí se puede mirar dónde marca agua la red. Sirve para el **análisis de fallos** (punto 5):
- El **mar** debería salir todo como agua (control de que la red funciona en Lima).
- Si aparecen manchas rojas en los **cerros**, es probable que sea **sombra de radar** (laderas que no reciben señal y se ven oscuras): el tipo de fallo que nos faltaba encontrar.
- Manchas en la **ciudad** serían superficies lisas (pistas, techos planos, el aeropuerto).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 9))
axes[0].imshow(img[0], cmap='gray', vmin=-25, vmax=0); axes[0].set_title('VV · 17-mar-2017')
axes[1].imshow(img[0], cmap='gray', vmin=-25, vmax=0)
axes[1].imshow(np.ma.masked_where(~pred_final, pred_final), cmap='autumn', alpha=0.8)
axes[1].contour(MASCARAS['Huella máxima'] == 1, levels=[0.5], colors='cyan', linewidths=0.6)
axes[1].contour(MASCARAS['Huella máxima'] != -1, levels=[0.5], colors='white', linewidths=0.8, linestyles='--')
axes[1].set_title('Rojo = agua según la red · cian = huella EMSN038 · blanco = AOI de EMSN038')
for ax in axes: ax.axis('off')
plt.tight_layout(); plt.show()

### C7. ¿Cuánto coinciden las 6 redes entre sí?

Para cada píxel se cuenta cuántos de los 6 modelos dicen "agua". Donde todos coinciden, la red está segura; donde solo 1–3 marcan agua, la respuesta **depende del azar del entrenamiento** (como vimos en Bolivia).

In [ ]:
votos = sum(p.astype('uint8') for p in PRED.values())
con_agua = votos > 0
print('Píxeles donde al menos una red marca agua:', int(con_agua.sum()))
print('De ellos, % donde coinciden las 6:', round(100 * (votos == 6).sum() / max(con_agua.sum(), 1), 1))

for zona, coords in ZONAS.items():
    f, c = ventana(*coords)
    plt.figure(figsize=(7, 5))
    plt.imshow(np.ma.masked_where(votos[f, c] == 0, votos[f, c]), cmap='viridis', vmin=1, vmax=6)
    plt.colorbar(label='nº de redes que dicen "agua"'); plt.title(zona); plt.axis('off'); plt.show()

## Qué anotar para el informe (Lima)

- [ ] **B2:** % de la huella que cae dentro de nuestra imagen (debería ser ≈ 100 %).
- [ ] **B3:** % de huella dentro del AOI. Comparar con Sen1Floods11 (≈ 10.6 % de agua).
- [ ] **B5:** ¿la huella cae sobre zonas oscuras del radar? (si no, el radar del 17 de marzo no ve agua ahí).
- [ ] **C3:** umbral de Otsu en Lima frente al rango de Sen1Floods11 (−19.6 a −25.1 dB).
- [ ] **C4:** IoU de las 2 recetas (media ± desv.) y de Otsu contra **huella** y contra **cauce**. Se espera IoU **bajo** contra la huella (son huellas, no agua visible) y mayor contra el cauce.
- [ ] **C4:** IoU por chip en Lima frente a test (0.400) y Bolivia (0.366).
- [ ] **C5–C6:** ¿hay **sombra de radar** en los cerros? Anotar una zona de ejemplo (es el 4.º tipo de fallo del punto 5).
- [ ] **C7:** % de acuerdo entre las 6 redes: ¿Lima es tan inestable como Bolivia?